# Stage Silver — INE Defunciones por Edad

Transforma las 3 tablas INE de defunciones por grupo etario hacia `stage_silver_ss2`.

| Tabla Bronze | Tabla Silver | Descripción |
|---|---|---|
| `ine_defunciones_sexo_edad_causas_muerte` | `ine_defunciones_sexo_edad` | General, todos los grupos de edad |
| `ine_defunciones_neonatales_sexo_edad_causas_muerte` | `ine_defunciones_neonatales` | 0-28 días (periodo neonatal) |
| `ine_defunciones_postneonatales_sexo_edad_causas_muerte` | `ine_defunciones_postneonatales` | 1-11 meses (post-neonatal) |

**Transformaciones aplicadas:**
- `anio`, `total`, `hombres`, `mujeres` → INT
- `cie_10_norm` = regexp_replace(codigo_cie_10, '[: ]', '') — elimina separadores; se conserva original
- Filas con `causa_de_muerte` IN ('Todas las causas', 'Otras causas') **se filtran** — son subtotales; Gold agrega desde datos granulares
- Validación de cuadratura (`total == hombres + mujeres`) — se loguea como warning si falla, sin columna en tabla
- `dropDuplicates()`
- Auditoría Silver: `silver_processed_timestamp`, `silver_job_run_id`

In [ ]:
# ── Setup: imports, config y helpers en una sola celda ────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

BRONZE_SCHEMA    = "sandbox_bronze_ss2"
SILVER_SCHEMA    = "stage_silver_ss2"
WRITE_MODE       = "overwrite"
_SUBTOTAL_LABELS = ["Todas las causas", "Otras causas"]

# ── Helpers ───────────────────────────────────────────────────────────────────

def get_job_run_id():
    try:
        return (
            dbutils.notebook.entry_point
            .getDbutils().notebook().getContext()
            .currentRunId().toString()
        )
    except Exception:
        return "manual"


def normalize_cie10(col_expr):
    """'Otras causas' → 'R99'  |  J:18:0 → J180  |  B:37 → B37  |  rangos como 'R00-R99' quedan igual"""
    return (
        F.when(col_expr.isin("Todas las causas", "Todas las causas externas"), F.lit("A00-Y98"))
         .when(col_expr == "Otras causas", F.lit("R99"))
         .otherwise(F.regexp_replace(col_expr, r'[: ]', ''))
    )


def validate_cuadratura(df, table_name):
    """Loguea filas donde total ≠ hombres + mujeres. No filtra — solo alerta."""
    fail = df.limit(5000).filter(F.col("total") != (F.col("hombres") + F.col("mujeres"))).count()
    if fail > 0:
        logger.warning(f"[{table_name}] {fail} filas (muestra 5k) con total ≠ H+M — revisar fuente")
    else:
        logger.info(f"[{table_name}] Cuadratura OK en muestra 5k")


def add_silver_audit(df, run_id):
    return (
        df
        .withColumn("silver_processed_timestamp", F.current_timestamp())
        .withColumn("silver_job_run_id",           F.lit(run_id))
    )


def process_ine_edad_table(bronze_tbl, silver_tbl, run_id):
    """
    Transformaciones Silver para las 3 tablas INE-edad.
    Esquema común: edad, codigo_cie_10, causa_de_muerte, total, hombres, mujeres, anio.
    Subtotales ('Todas las causas', 'Otras causas') se filtran — Gold agrega desde granular.
    """
    short = bronze_tbl.split(".")[-1]
    df = spark.table(bronze_tbl)
    logger.info(f"[{short}] Bronze: {df.count():,} filas")

    df_silver = (
        df
        .withColumn("anio",        F.col("anio").cast(IntegerType()))
        .withColumn("total",       F.col("total").cast(IntegerType()))
        .withColumn("hombres",     F.col("hombres").cast(IntegerType()))
        .withColumn("mujeres",     F.col("mujeres").cast(IntegerType()))
        .withColumn("cie_10_norm", normalize_cie10(F.col("codigo_cie_10")))
        # Filtrar filas de subtotal — no son datos granulares
        .filter(~F.col("causa_de_muerte").isin(_SUBTOTAL_LABELS))
        .dropDuplicates()
    )

    validate_cuadratura(df_silver, short)
    df_silver = add_silver_audit(df_silver, run_id)

    logger.info(f"[{short}] Silver: {df_silver.count():,} filas")
    (
        df_silver.write
        .format("delta")
        .mode(WRITE_MODE)
        .option("overwriteSchema", "true")
        .saveAsTable(silver_tbl)
    )
    logger.info(f"Escrito → {silver_tbl}")


RUN_ID = get_job_run_id()
logger.info(f"Setup OK — run_id={RUN_ID}")

## Tabla 1: ine_defunciones_sexo_edad_causas_muerte → ine_defunciones_sexo_edad

In [ ]:
process_ine_edad_table(
    bronze_tbl=f"{BRONZE_SCHEMA}.ine_defunciones_sexo_edad_causas_muerte",
    silver_tbl=f"{SILVER_SCHEMA}.ine_defunciones_sexo_edad",
    run_id=RUN_ID,
)

## Tabla 2: ine_defunciones_neonatales_sexo_edad_causas_muerte → ine_defunciones_neonatales
Grupo etario: días (0-28 días). Incluye códigos de rango como `R00 - R99`.

In [ ]:
process_ine_edad_table(
    bronze_tbl=f"{BRONZE_SCHEMA}.ine_defunciones_neonatales_sexo_edad_causas_muerte",
    silver_tbl=f"{SILVER_SCHEMA}.ine_defunciones_neonatales",
    run_id=RUN_ID,
)

## Tabla 3: ine_defunciones_postneonatales_sexo_edad_causas_muerte → ine_defunciones_postneonatales
Grupo etario: meses (1-11 meses).

In [ ]:
process_ine_edad_table(
    bronze_tbl=f"{BRONZE_SCHEMA}.ine_defunciones_postneonatales_sexo_edad_causas_muerte",
    silver_tbl=f"{SILVER_SCHEMA}.ine_defunciones_postneonatales",
    run_id=RUN_ID,
)

## Validación — Row counts, cuadratura y subtotales

In [ ]:
_MAP = [
    ("ine_defunciones_sexo_edad_causas_muerte",               "ine_defunciones_sexo_edad"),
    ("ine_defunciones_neonatales_sexo_edad_causas_muerte",    "ine_defunciones_neonatales"),
    ("ine_defunciones_postneonatales_sexo_edad_causas_muerte","ine_defunciones_postneonatales"),
]

print(f"{'Tabla Silver':<40} {'Bronze':>8} {'Silver':>8} {'Filtradas':>10}")
print("-" * 70)
for bronze_name, silver_name in _MAP:
    b_cnt = spark.table(f"{BRONZE_SCHEMA}.{bronze_name}").count()
    s_cnt = spark.table(f"{SILVER_SCHEMA}.{silver_name}").count()
    filtradas = b_cnt - s_cnt
    print(f"{silver_name:<40} {b_cnt:>8,} {s_cnt:>8,} {filtradas:>10,}")

print("\nNota: 'Filtradas' = subtotales ('Todas las causas', 'Otras causas') eliminados en Silver.")

print("\n── Muestra ine_defunciones_sexo_edad (limit 5) ──")
(
    spark.table(f"{SILVER_SCHEMA}.ine_defunciones_sexo_edad")
    .select("anio", "edad", "causa_de_muerte", "codigo_cie_10", "cie_10_norm", "total", "hombres", "mujeres")
    .limit(5)
    .show(truncate=False)
)

print("── Departamentos sin match (no aplica a estas tablas — sin columna de depto) ──")
print("   Las tablas INE-edad no tienen dimensión geográfica.")

print("\n── Verificar que no quedan subtotales en Silver ──")
for bronze_name, silver_name in _MAP:
    remaining = (
        spark.table(f"{SILVER_SCHEMA}.{silver_name}")
        .filter(F.col("causa_de_muerte").isin(["Todas las causas", "Otras causas"]))
        .limit(10)
        .count()
    )
    status = "OK — sin subtotales" if remaining == 0 else f"WARN — {remaining} subtotales encontrados"
    print(f"  {silver_name}: {status}")